In [5]:
import sys
import os

project_root = os.path.abspath("..")
sys.path.append(project_root)

In [14]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Reload dataset
df = pd.read_csv('../data/raw/application_train.csv')

X = df.drop(columns=['TARGET', 'SK_ID_CURR'])
y = df['TARGET']

# SAME split used during training
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Create df_test
df_test = X_test.copy()
df_test['actual'] = y_test.values

In [18]:
import mlflow.sklearn
model = mlflow.sklearn.load_model("models:/credit-risk-lgbm/2")

In [19]:
from fairlearn.metrics import MetricFrame, selection_rate, false_positive_rate
from sklearn.metrics import accuracy_score

# Create age in years
df_test['AGE_YEARS'] = (-df_test['DAYS_BIRTH']) / 365

# Age bands
df_test['age_band'] = pd.cut(
    df_test['AGE_YEARS'],
    bins=[18, 30, 45, 60, 100],
    labels=['18-30', '31-45', '46-60', '60+']
)

# Predictions
df_test['predicted'] = (model.predict_proba(X_test)[:, 1] > 0.3).astype(int)

# Fairness metrics
metric_frame = MetricFrame(
    metrics={
        'accuracy': accuracy_score,
        'selection_rate': selection_rate,
        'fpr': false_positive_rate,
    },
    y_true=y_test,
    y_pred=df_test['predicted'],
    sensitive_features=df_test['age_band']
)

print("=== Fairness Metrics by Age Group ===")
print(metric_frame.by_group)
print()
print("Max Accuracy Disparity:", metric_frame.difference()["accuracy"])

=== Fairness Metrics by Age Group ===
          accuracy  selection_rate       fpr
age_band                                    
18-30     0.380329        0.716670  0.689598
31-45     0.488440        0.579628  0.550009
46-60     0.574896        0.466040  0.441554
60+       0.660234        0.358511  0.341262

Max Accuracy Disparity: 0.27990497308653867
